# Working with ChatGPT

> The .py file will be populated from multiple notebooks by using


In [1]:
# | default_exp chat

In [2]:
# | hide
import nbdev
import dotenv

dotenv.load_dotenv()
nbdev.nbdev_export()

In [3]:
# | export
import pydantic
from pydantic import BaseModel, Field
from openai import ChatCompletion

from lovely_prompts.schema import *

In [4]:
# | export

assert pydantic.__version__.startswith("1.")  # We will have to do a small rewrite fro 2.0


class BaseMessage(StrictBaseModel):
    content: str = Field(None, description="The content of the message")
    role: str = Field(None, description="The role of the message")


class SystemMessage(BaseMessage):
    role: str = "system"


class AssistantMessage(BaseMessage):
    role: str = "assistant"


class UserMessage(BaseMessage):
    role: str = "user"

In [5]:
test_prompt = [
    UserMessage(content="Hello, I am a human"),
    AssistantMessage(content="Hello, I am an AI"),
]

In [6]:
[t.dict() for t in test_prompt]

[{'content': 'Hello, I am a human', 'role': 'user'},
 {'content': 'Hello, I am an AI', 'role': 'assistant'}]

In [7]:
# | exporti


class _OpenAIUsage(LaxBaseModel):
    prompt_tokens: int
    completion_tokens: int


class _OpenAIChoice(LaxBaseModel):
    index: int
    message: BaseMessage
    finish_reason: str


class _OpenAIChatResponse(LaxBaseModel):
    choices: list[_OpenAIChoice]
    usage: _OpenAIUsage = Field(..., description="The token usage of the chat completion")

In [8]:
# | export


class ResponseMessage(BaseMessage):
    tok_in: int = Field(None, description="The number of tokens in the prompt")
    tok_out: int = Field(None, description="The number of tokens in the completion")
    finish_reason: str = Field(None, description="The reason the chat was finished")
    model: str = Field(None, description="The model used to generate the response")
    id: str = Field(None, description="The id of the chat completion")
    created: int = Field(None, description="The time the chat was created, UNIX timestamp")

In [9]:
# | exporti
from tiktoken import encoding_for_model, Encoding

In [10]:
# | export


class ChatOpenAI(BaseModel):
    class Config:
        underscore_attrs_are_private = True

    model: str = Field(..., description="The model to use for the chat", frozen=True)
    temperature: float = Field(0.7, description="The temperature of the chat", frozen=True)

    _encoding: Encoding | None = None

    def generate(
        self, prompt: list[BaseMessage] | str, n=1, **kwargs
    ) -> str | list[str] | ResponseMessage | list[ResponseMessage]:
        return_string = False
        if isinstance(prompt, str):
            return_string = True
            prompt = [UserMessage(content=prompt)]

        res = ChatCompletion().create(
            model=self.model,
            temperature=self.temperature,
            messages=[p.dict() for p in prompt],
            n=n,
            **kwargs,
        )
        res = _OpenAIChatResponse(**res)

        assert len(res.choices) == n

        if return_string and n == 1:
            return res.choices[0].message.content
        if return_string:
            return [choice.message.content for choice in res.choices]

        response_messages = [
            ResponseMessage(
                role=choice.message.role,
                content=choice.message.content,
                finish_reason=choice.finish_reason,
                model=res.model,
                id=res.id,
                created=res.created,
            )
            for choice in res.choices
        ]

        if n == 1:
            # The token counts are only relevant for a single response, we don't get them per-message.
            response_messages[0].tok_in = res.usage.prompt_tokens
            response_messages[0].tok_out = res.usage.completion_tokens
            return response_messages[0]
        return response_messages

    # Simplified verson from the OpenAI cookbook
    def num_tokens_from_messages(self, messages: list[BaseMessage]) -> int:
        """Return the number of tokens used by a list of messages."""
        if self._encoding is None:
            self._encoding = encoding_for_model(self.model)

        tokens_per_message = 4

        num_tokens = 0
        for message in messages:
            num_tokens += tokens_per_message
            num_tokens += len(self._encoding.encode(message.content)) + 4  # +4 for role (1 token) + special tokens
        num_tokens += 3  # every reply is primed with <|start|>assistant<|message|>
        return num_tokens

    def num_tokens_from_string(self, prompt: str) -> int:
        """Return the number of tokens used by a string prompt."""
        return self.num_tokens_from_messages([UserMessage(content=prompt)])

    def __call__(self, prompt: str | list[BaseMessage], n=1, **kwargs):
        return self.generate(prompt, n=n, **kwargs)

In [11]:
# | eval: false
chat = ChatOpenAI(model="gpt-3.5-turbo")

In [12]:
# | eval: false
chat("Hello there")

'Hello! How can I assist you today?'

In [13]:
# | eval: false
chat("Hello there", n=2)

['General Kenobi!', 'General Kenobi!']

In [14]:
test_prompt = [
    UserMessage(content="Hello, I am a human"),
    AssistantMessage(content="Hello, I am an AI"),
]

In [15]:
# | eval: false
chat(test_prompt)

ResponseMessage(content='Nice to meet you, fellow human! How can I assist you today?', role='assistant', tok_in=23, tok_out=15, finish_reason='stop', model='gpt-3.5-turbo-0613', id='chatcmpl-7bSg9jT51QWAgs0VTwTkgk4o7sTnn', created=1689162733)

In [16]:
# | eval: false
chat(test_prompt, n=2)

[ResponseMessage(content='Nice to meet you, human! How can I assist you today?', role='assistant', tok_in=None, tok_out=None, finish_reason='stop', model='gpt-3.5-turbo-0613', id='chatcmpl-7bSgAIEqjKuZt4L1k79uFUOm9jTOB', created=1689162734),
 ResponseMessage(content='Nice to meet you, fellow human! How can I assist you today?', role='assistant', tok_in=None, tok_out=None, finish_reason='stop', model='gpt-3.5-turbo-0613', id='chatcmpl-7bSgAIEqjKuZt4L1k79uFUOm9jTOB', created=1689162734)]

In [17]:
chat.num_tokens_from_messages(test_prompt)

31